In [6]:
from google.colab import auth
auth.authenticate_user()              # 弹出授权窗口，选拥有 GCP 项目的那个 Google 账号

from google.cloud import bigquery

PROJECT = "serene-essence-508805-b2"           # ← 改成你的项目 ID，只用来跑查询
DATASET = "bigquery-public-data.thelook_ecommerce"
CUTOFF  = "2026-09-01"                # 冻结点：只保留此日期之前创建的行，8 月是最后一个完整月

bq = bigquery.Client(project=PROJECT)


In [7]:
print(bq.project)
print(bq.query("SELECT 1 AS ok").to_dataframe())

serene-essence-508805-b2
   ok
0   1


In [8]:
import json, hashlib, datetime, pathlib, shutil
import pandas as pd, pyarrow

out = pathlib.Path("snapshot"); out.mkdir(exist_ok=True)
manifest = {
    "source": DATASET, "cutoff": CUTOFF,
    "exported_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "versions": {"pandas": pd.__version__, "pyarrow": pyarrow.__version__},
    "bytes_processed": 0, "tables": {}, "checks": {},
}

def run(sql):
    job = bq.query(sql)
    df = job.to_dataframe()
    manifest["bytes_processed"] += job.total_bytes_processed or 0
    return df

def table_columns(table):             # 列清单从元数据读，GEOGRAPHY 列丢掉
    df = run(f"""SELECT column_name, data_type
                 FROM `{DATASET}.INFORMATION_SCHEMA.COLUMNS`
                 WHERE table_name = '{table}' ORDER BY ordinal_position""")
    if df.empty:
        raise RuntimeError(f"table {table!r} not found in {DATASET}")
    return [r.column_name for r in df.itertuples() if r.data_type != "GEOGRAPHY"]

def save(name, sql):
    df = run(sql)
    path = out / f"{name}.parquet"
    df.to_parquet(path, index=False)
    manifest["tables"][name] = {
        "rows": int(len(df)), "columns": list(df.columns),
        "file_bytes": path.stat().st_size,
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }
    print(f"{name:<22}{len(df):>11,} rows{path.stat().st_size / 1e6:>9.1f} MB")
    return df


In [9]:
T = lambda name: f"`{DATASET}.{name}`"
before = f"created_at < TIMESTAMP('{CUTOFF}')"
kept_orders = f"SELECT order_id FROM {T('orders')} WHERE {before}"

WHERE = {
    "distribution_centers": "",
    "products": "",
    "orders": f"WHERE {before}",
    "order_items": f"WHERE order_id IN ({kept_orders})",          # 跟着订单走，不单独按时间筛
    "users": f"WHERE {before} OR id IN (SELECT user_id FROM {T('orders')} WHERE {before})",
    "inventory_items": f"WHERE {before} OR id IN (SELECT inventory_item_id FROM {T('order_items')} "
                       f"WHERE order_id IN ({kept_orders}))",
}
frames = {}
for name, where in WHERE.items():
    select = ", ".join(f"`{c}`" for c in table_columns(name))
    frames[name] = save(name, f"SELECT {select} FROM {T(name)} {where} ORDER BY 1")   # 固定行序


distribution_centers           10 rows      0.0 MB
products                   29,120 rows      2.3 MB
orders                    117,114 rows      3.8 MB
order_items               170,036 rows      7.1 MB
users                      96,872 rows      5.9 MB
inventory_items           484,911 rows     20.6 MB


In [10]:
ev = set(table_columns("events"))
missing = {"session_id", "created_at", "event_type"} - ev
assert not missing, f"events is missing expected columns: {missing}. Columns found: {sorted(ev)}"

types = run(f"SELECT event_type, COUNT(*) AS n FROM {T('events')} GROUP BY 1 ORDER BY n DESC")
print(types.to_string(index=False), "\n")
found = set(types["event_type"])
for must in ("product", "cart", "purchase"):
    assert must in found, f"event_type {must!r} not found; adjust the funnel flags below. Found: {sorted(found)}"
manifest["checks"]["event_types"] = {r.event_type: int(r.n) for r in types.itertuples()}

dims = [d for d in ("user_id", "traffic_source", "browser", "state") if d in ev]
dim_sql = "".join(f"  ANY_VALUE(`{d}`) AS `{d}`,\n" for d in dims)
frames["sessions"] = save("sessions", f"""
SELECT
  session_id,
{dim_sql}  MIN(created_at) AS started_at,
  COUNT(*) AS n_events,
  LOGICAL_OR(event_type = 'product')  AS saw_product,
  LOGICAL_OR(event_type = 'cart')     AS added_to_cart,
  LOGICAL_OR(event_type = 'purchase') AS purchased
FROM {T('events')}
WHERE {before}
GROUP BY session_id
ORDER BY session_id""")


event_type      n
   product 843938
      cart 594376
department 593336
  purchase 180858
    cancel 125623
      home  87222 

sessions                  667,213 rows     30.1 MB


In [11]:
o, oi, u, p, s = (frames[k] for k in ("orders", "order_items", "users", "products", "sessions"))
cutoff_ts = pd.Timestamp(CUTOFF, tz="UTC")
assert o["created_at"].max() < cutoff_ts, "orders contain rows at or after the cutoff"

manifest["checks"].update({
    "max_order_created_at": str(o["created_at"].max()),
    "order_items_without_order": int((~oi["order_id"].isin(o["order_id"])).sum()),
    "order_items_without_product": int((~oi["product_id"].isin(p["id"])).sum()),
    "orders_without_user": int((~o["user_id"].isin(u["id"])).sum()),
    "session_purchase_rate": round(float(s["purchased"].mean()), 4),
})
monthly = (oi.assign(month=oi["created_at"].dt.strftime("%Y-%m"))
             .groupby("month").agg(items=("id", "count"), revenue=("sale_price", "sum")).tail(8))
manifest["checks"]["monthly_tail"] = {m: {"items": int(r["items"]), "revenue": round(float(r["revenue"]), 2)}
                                      for m, r in monthly.iterrows()}
print(monthly.round(0).to_string(), "\n")
for k, v in manifest["checks"].items():
    if k not in ("event_types", "monthly_tail"):
        print(f"{k:<30}{v}")
print(f"\nBigQuery bytes processed: {manifest['bytes_processed'] / 1e9:.2f} GB of the 1,000 GB monthly free tier")

(out / "manifest.json").write_text(json.dumps(manifest, indent=2))
shutil.make_archive("snapshot", "zip", root_dir=".", base_dir="snapshot")

from google.colab import files
files.download("snapshot.zip")


         items   revenue
month                   
2026-02   4522  272342.0
2026-03   5439  326029.0
2026-04   5425  328109.0
2026-05   6068  358467.0
2026-06   6611  387568.0
2026-07   7312  429079.0
2026-08   8851  516172.0
2026-09    295   17009.0 

max_order_created_at          2026-08-31 23:44:11+00:00
order_items_without_order     0
order_items_without_product   0
orders_without_user           0
session_purchase_rate         0.2544

BigQuery bytes processed: 0.44 GB of the 1,000 GB monthly free tier


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>